[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/editorial-v2/notebooks/05_State_Space_Models.ipynb)

# DiveLab

## Notebook 05 — State-Space Models

**From motion to state**

### Guiding question

What information must be known now to predict a diver's future vertical motion?

This notebook turns the coupled equations from Chapter 4 into an executable nonlinear state-space model. It is an educational model, not an operational tool for planning or controlling a dive.

## Learning objectives

By the end of this notebook, you should be able to:

- construct the state vector $\mathbf{x}=[z,v]^{\mathsf T}$;
- distinguish states from algebraic variables, inputs, disturbances, parameters and outputs;
- implement $\dot{\mathbf{x}}=\mathbf{f}(t,\mathbf{x},u,d)$ in Python;
- evaluate the derivative at an operating state;
- integrate and interpret autonomous and forced trajectories;
- explain why a nonlinear state-space model is not automatically linear.

## From Notebook 04 to Notebook 05

Notebook 04 assembled the vertical equations

$$
\dot{z}=-v,
$$

$$
m\dot{v}=F_B(z)-mg-cv|v|.
$$

Here $z$ is depth, positive downward, and $v$ is vertical velocity, positive upward. Notebook 05 does not introduce new diving physics. It reorganizes the existing model so that it can be simulated and later analyzed.

## What makes a variable a state?

The state carries the information needed to continue the model into the future once future inputs are specified. Depth alone is insufficient: two divers at the same depth can have different velocities and therefore different immediate futures.

For this model,

$$
\mathbf{x}(t)=
\begin{bmatrix}
z(t)\\
v(t)
\end{bmatrix}.
$$

Pressure, gas volume, buoyancy and drag are calculated immediately from $z$, $v$ and the parameters. They are algebraic variables, not additional states.

## Model conventions and assumptions

- Depth $z\geq0$ increases downward.
- Velocity $v$ is positive upward, so $\dot{z}=-v$.
- Water density is constant.
- Boyle's law is isothermal and instantaneous.
- Fixed displaced volume is incompressible.
- Drag is quadratic and acts opposite velocity.
- The control input and disturbance are represented as vertical forces.
- Gas exchange, valve dynamics, breathing cycles, added mass and orientation changes are omitted.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

## Parameters

Parameters remain fixed during one simulation. SI units are used internally.

In [ ]:
rho = 1025.0          # seawater density [kg/m^3]
g = 9.80665           # gravitational acceleration [m/s^2]
p0 = 101_325.0        # surface absolute pressure [Pa]
mass = 85.0           # diver-and-equipment mass [kg]
surface_gas_volume = 8.0e-3  # flexible gas volume at surface [m^3]
drag_coefficient = 0.80
projected_area = 0.70  # [m^2]

drag_parameter = 0.5 * rho * drag_coefficient * projected_area

print(f"Drag parameter: {drag_parameter:.1f} kg/m")

## Algebraic relations

The current state determines pressure, flexible gas volume, buoyancy and drag:

$$
P_{\mathrm{abs}}(z)=P_0+\rho gz,
$$

$$
V_g(z)=V_{g0}\frac{P_0}{P_{\mathrm{abs}}(z)},
$$

$$
F_B(z)=\rho g\left[V_f+V_g(z)\right],
$$

$$
F_D(v)=-cv|v|.
$$

In [ ]:
def ambient_pressure(depth_m):
    # Absolute ambient pressure [Pa].
    return p0 + rho * g * depth_m


def gas_volume(depth_m):
    # Flexible gas volume [m^3] from Boyle's law.
    return surface_gas_volume * p0 / ambient_pressure(depth_m)


def buoyant_force(depth_m, fixed_volume_m3):
    # Total upward buoyant force [N].
    return rho * g * (fixed_volume_m3 + gas_volume(depth_m))


def drag_force(velocity_m_s):
    # Signed drag force [N], positive upward.
    return -drag_parameter * velocity_m_s * abs(velocity_m_s)

## Construct a neutral operating state

Choose the fixed displaced volume so that buoyancy equals weight at the target depth $z^*=20\ \mathrm{m}$:

$$
V_f=\frac{m}{\rho}-V_g(z^*).
$$

In [ ]:
neutral_depth = 20.0  # [m]
fixed_volume = mass / rho - gas_volume(neutral_depth)

print(f"Neutral depth:          {neutral_depth:.1f} m")
print(f"Flexible gas volume:    {gas_volume(neutral_depth) * 1000:.3f} L")
print(f"Fixed displaced volume: {fixed_volume * 1000:.3f} L")
print(
    "Force residual:        "
    f"{buoyant_force(neutral_depth, fixed_volume) - mass * g:+.3e} N"
)

## Add input and disturbance channels

Let $u(t)$ be a commanded upward force and $d(t)$ an uncommanded upward disturbance. Their units and mathematical location are identical, but their roles differ.

The nonlinear state equation is

$$
\dot{\mathbf{x}}=
\begin{bmatrix}
-v\\[4pt]
\dfrac{F_B(z)-mg-cv|v|+u+d}{m}
\end{bmatrix}.
$$

In [ ]:
def signal_value(signal, time_s, state):
    # Evaluate a constant or callable signal.
    return signal(time_s, state) if callable(signal) else float(signal)


def state_derivative(
    time_s,
    state,
    control_force=0.0,
    disturbance_force=0.0,
):
    # Return [depth_rate, upward_acceleration] in SI units.
    depth_m, velocity_m_s = state
    control = signal_value(control_force, time_s, state)
    disturbance = signal_value(disturbance_force, time_s, state)

    net_force = (
        buoyant_force(depth_m, fixed_volume)
        - mass * g
        + drag_force(velocity_m_s)
        + control
        + disturbance
    )

    depth_rate = -velocity_m_s
    acceleration = net_force / mass
    return np.array([depth_rate, acceleration])

## Evaluate the derivative at the equilibrium

In [ ]:
equilibrium_state = np.array([neutral_depth, 0.0])
equilibrium_derivative = state_derivative(0.0, equilibrium_state)

print("State [depth, upward velocity]:", equilibrium_state)
print("Derivative [depth rate, acceleration]:", equilibrium_derivative)

Both derivative components are zero: the system is stationary and neutrally buoyant. This identifies an equilibrium candidate. Chapter 6 will test its stability.

## Same depth, different states

In [ ]:
states_at_same_depth = {
    "descending": np.array([neutral_depth, -0.20]),
    "stationary": np.array([neutral_depth, 0.00]),
    "ascending": np.array([neutral_depth, 0.20]),
}

for label, state in states_at_same_depth.items():
    derivative = state_derivative(0.0, state)
    print(
        f"{label:10s}  state={state}  "
        f"derivative={np.round(derivative, 5)}"
    )

The three cases share the same depth and buoyant force, but their depth rates and drag forces differ. This is why depth alone is not a sufficient state description.

## One explicit Euler step

In [ ]:
time_step = 0.50  # [s]
initial_state = np.array([20.0, 0.10])
initial_derivative = state_derivative(0.0, initial_state)
euler_state = initial_state + time_step * initial_derivative

print("Initial state: ", np.round(initial_state, 5))
print("Derivative:    ", np.round(initial_derivative, 5))
print("Euler estimate:", np.round(euler_state, 5))

Euler's method exposes the update logic,

$$
\mathbf{x}_{k+1}\approx\mathbf{x}_k+\Delta t\,\dot{\mathbf{x}}_k,
$$

but the experiments below use `solve_ivp`, which provides a more accurate adaptive integration method.

## A reusable simulation function

In [ ]:
def simulate(
    initial_state,
    duration_s=25.0,
    control_force=0.0,
    disturbance_force=0.0,
    sample_count=501,
):
    # Integrate the nonlinear state equation.
    sample_times = np.linspace(0.0, duration_s, sample_count)

    solution = solve_ivp(
        fun=lambda t, x: state_derivative(
            t,
            x,
            control_force=control_force,
            disturbance_force=disturbance_force,
        ),
        t_span=(0.0, duration_s),
        y0=np.asarray(initial_state, dtype=float),
        t_eval=sample_times,
        rtol=1e-8,
        atol=1e-10,
    )

    if not solution.success:
        raise RuntimeError(solution.message)
    return solution

## Autonomous trajectories

Set $u=d=0$ and release the model from rest $0.5\ \mathrm{m}$ above and below the neutral depth. These are two different initial conditions for the same autonomous system.

In [ ]:
initial_conditions = {
    "0.5 m above neutral": np.array([neutral_depth - 0.5, 0.0]),
    "equilibrium": equilibrium_state,
    "0.5 m below neutral": np.array([neutral_depth + 0.5, 0.0]),
}

autonomous_solutions = {
    label: simulate(state)
    for label, state in initial_conditions.items()
}

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

for label, solution in autonomous_solutions.items():
    axes[0].plot(solution.t, solution.y[0], label=label)
    axes[1].plot(solution.t, solution.y[1], label=label)

axes[0].axhline(neutral_depth, color="black", linestyle="--", linewidth=1)
axes[0].set_ylabel("Depth [m]")
axes[0].set_title("Autonomous nonlinear state trajectories")
axes[0].invert_yaxis()
axes[0].grid(True)
axes[0].legend()

axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Upward velocity [m/s]")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

## Interpretation

The equilibrium trajectory remains at $z^*$ and $v=0$ in the ideal numerical model. The displaced trajectories evolve because flexible-gas compression couples depth back into buoyancy.

The plots show state components against time. They do **not** yet constitute a formal stability analysis. In Chapter 6 we will linearize near the equilibrium and plot velocity directly against depth to expose the phase-plane geometry.

## A forced trajectory

Starting from equilibrium, apply a small upward force for the first three seconds:

$$
u(t)=
\begin{cases}
5\ \mathrm{N}, & 0\leq t<3\ \mathrm{s},\\
0, & t\geq3\ \mathrm{s}.
\end{cases}
$$

This is an abstract force input. It is not yet a physical BCD or diver controller.

In [ ]:
def force_pulse(time_s, state):
    return 5.0 if time_s < 3.0 else 0.0


forced_solution = simulate(
    equilibrium_state,
    duration_s=25.0,
    control_force=force_pulse,
)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)

axes[0].plot(forced_solution.t, forced_solution.y[0], color="tab:blue")
axes[0].axhline(neutral_depth, color="black", linestyle="--", linewidth=1)
axes[0].set_ylabel("Depth [m]")
axes[0].set_title("Response to a three-second upward force pulse")
axes[0].invert_yaxis()
axes[0].grid(True)

axes[1].plot(forced_solution.t, forced_solution.y[1], color="tab:orange")
axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].set_ylabel("Upward velocity [m/s]")
axes[1].grid(True)

input_history = np.where(forced_solution.t < 3.0, 5.0, 0.0)
axes[2].step(forced_solution.t, input_history, where="post", color="tab:green")
axes[2].set_xlabel("Time [s]")
axes[2].set_ylabel("Input force [N]")
axes[2].grid(True)

plt.tight_layout()
plt.show()

## Interpretation

The input acts for only three seconds, but it changes the state. When the pulse ends, the system does not return automatically to its previous state: the new depth and velocity become the initial condition for the subsequent autonomous motion.

This separation between **input history** and **stored state** is one of the main reasons state-space models are useful.

## Output equations

An output specifies what the model exposes. The full model state can be returned directly, or a depth-only output can represent an idealized depth measurement.

In [ ]:
def full_state_output(state):
    return np.asarray(state, dtype=float)


def depth_output(state):
    return float(state[0])


sample_state = np.array([18.0, 0.25])
print("Full-state output:", full_state_output(sample_state))
print("Depth-only output:", depth_output(sample_state), "m")

The state exists whether or not every component is measured. Later, a model may use depth measurements to estimate vertical velocity. That is an estimation problem, not a reason to remove velocity from the state.

## State-space does not mean linear

The notation

$$
\dot{\mathbf{x}}=\mathbf{f}(\mathbf{x},u,d)
$$

describes a state-space model without assuming linearity. The present $\mathbf{f}$ contains $1/(P_0+\rho gz)$ and $v|v|$, so it is nonlinear.

The linear form

$$
\dot{\mathbf{x}}=A\mathbf{x}+B\mathbf{u}
$$

is a special case. Chapter 6 will derive a local linear model near the neutral operating state.

## Exercises

### 1. Inspect another state

Evaluate the derivative at $z=15\ \mathrm{m}$ and $v=-0.15\ \mathrm{m\,s^{-1}}$. Interpret the sign of each component.

In [ ]:
# Your code here

### 2. Change the initial velocity

Start at the neutral depth with $v=+0.10\ \mathrm{m\,s^{-1}}$ and then with $v=-0.10\ \mathrm{m\,s^{-1}}$. Plot both autonomous responses and explain why they differ.

In [ ]:
# Your code here

### 3. Apply a disturbance

Replace the commanded input by a two-second downward disturbance of $-4\ \mathrm{N}$. Compare the response with the upward-force experiment.

In [ ]:
# Your code here

### 4. Compare output choices

For one simulated trajectory, construct:

1. a depth-only output;
2. a velocity-only output;
3. a full-state output.

Explain which variables remain internal in each case.

In [ ]:
# Your code here

## Challenge — add actuator memory

The current input $u(t)$ changes force instantaneously. Propose a third state $a(t)$ representing an actuator force with first-order dynamics,

$$
\tau\dot{a}=u_c-a,
$$

and replace $u$ in the vertical force balance by $a$. Write the resulting three-state vector and derivative function. Explain why $a$ is now a state rather than an algebraic variable.

## Summary

- The minimal state of the current vertical model is $\mathbf{x}=[z,v]^{\mathsf T}$.
- Pressure, gas volume, buoyancy and drag are algebraic functions of the state.
- Inputs are commanded; disturbances are uncommanded; parameters remain fixed during one simulation.
- The state derivative defines a local direction of evolution.
- Initial conditions and input histories select trajectories.
- State-space notation applies to nonlinear as well as linear systems.
- A variable becomes an additional state when its own memory or dynamics must be modeled.

## Next notebook

Notebook 06 will study equilibria and stability. It will recover the phase-plane material formerly attached to Notebook 03 and place it after the required state-space foundation: local linearization, eigenvalues, vector fields and the nonlinear phase portrait.